# Donor Clustering

This notebook groups donors with **K-Means clustering** on three behavior features, then compares the clusters with the rule-based segments from `src/donor_analysis.py`.

- **Recency:** days since the last gift
- **Frequency:** number of gifts
- **Monetary:** lifetime giving

All data is synthetic (`data/donors.csv`, 100 donors). The same logic runs from the command line with `python src/donor_clustering.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from donor_analysis import INACTIVE_DAYS, load_donors, segment
from donor_clustering import (
    DEFAULT_K, FEATURES, cluster, compare_with_segments, profile, rfm_features, score_k,
)

# Chart style: one accent color, gray for context, quiet axes and grid.
ACCENT = "#2a78d6"
CONTEXT = "#c3c2b7"
SURFACE = "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e",
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0b0b0b",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "xtick.color": "#898781",
    "ytick.color": "#898781",
    "font.size": 10,
})
dollars = FuncFormatter(lambda v, _: f"${v:,.0f}")

LABELS = {
    "days_since_last_donation": "Days since last gift",
    "donation_count": "Number of gifts",
    "total_donation": "Lifetime giving ($)",
}

df = load_donors(ROOT / "data" / "donors.csv")
df.head()

## 1. Features

K-Means measures straight-line distance, so the features need to be on comparable scales. All three are right-skewed: most donors give a little, a few give a lot. Two steps fix this before clustering:

1. **Log transform** (`log1p`) pulls in the long tails, so a few large donors do not dominate.
2. **Standardize** each feature to mean 0 and standard deviation 1, so each counts equally.

In [ ]:
df[FEATURES].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, FEATURES):
    ax.hist(df[col], bins=15, color=ACCENT, edgecolor=SURFACE, linewidth=1.5)
    ax.set_title(LABELS[col])
    ax.set_ylabel("Donors" if col == FEATURES[0] else "")
    ax.grid(axis="x", visible=False)
axes[2].xaxis.set_major_formatter(dollars)
fig.suptitle("Raw feature distributions: all three are right-skewed", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 2. Choosing the number of clusters (k)

Two common checks:

- **Inertia** (elbow method): total distance from donors to their cluster center. It always falls as k grows; look for the point where the drop slows.
- **Silhouette score:** how well separated the clusters are, from -1 to 1. Higher is better.

In [ ]:
X = rfm_features(df)
scores = score_k(X)
scores.round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True)
for ax, col, title in [
    (ax1, "inertia", "Inertia (lower = tighter clusters)"),
    (ax2, "silhouette", "Silhouette score (higher = better separated)"),
]:
    ax.plot(scores.index, scores[col], color=ACCENT, linewidth=2, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5)
    ax.axvline(DEFAULT_K, color="#898781", linewidth=1, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Number of clusters (k)")
ax1.annotate(f"k = {DEFAULT_K} (chosen)", xy=(DEFAULT_K, scores.loc[DEFAULT_K, "inertia"]),
             xytext=(8, 8), textcoords="offset points", color="#52514e")
ax2.set_ylim(0, 0.5)
fig.tight_layout()
plt.show()

There is no sharp elbow, and silhouette scores for k = 2 to 5 sit close together (about 0.33 to 0.39). That is typical for smooth synthetic data: donors spread along a continuum rather than forming separate islands.

Because no k is a clear statistical winner, this notebook uses **k = 4**: it is still in the good silhouette range and lines up with the four rule-based segments, which makes the two approaches easy to compare.

## 3. Cluster profiles

Clusters are numbered so that **cluster 1 has the highest median lifetime giving**.

In [ ]:
df["cluster"] = cluster(df, DEFAULT_K)
df["segment"] = segment(df)

profiles = profile(df)
profiles.round({"median_days_since_last": 0, "median_gifts": 0, "median_total": 0,
                "share_of_giving": 2})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, (c, row) in zip(axes.flat, profiles.iterrows()):
    others = df[df["cluster"] != c]
    members = df[df["cluster"] == c]
    ax.scatter(others["days_since_last_donation"], others["total_donation"],
               s=18, color=CONTEXT, alpha=0.6, linewidths=0)
    ax.scatter(members["days_since_last_donation"], members["total_donation"],
               s=40, color=ACCENT, edgecolor=SURFACE, linewidth=1)
    ax.axvline(INACTIVE_DAYS, color="#898781", linewidth=1, linestyle="--")
    ax.set_title(f"Cluster {c}: {row['donors']:.0f} donors, {row['share_of_giving']:.0%} of giving",
                 loc="left")
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(dollars)
axes[0, 0].annotate(f"Inactive after {INACTIVE_DAYS} days", xy=(INACTIVE_DAYS, 1500),
                    xytext=(-6, 0), textcoords="offset points", ha="right", color="#52514e")
for ax in axes[1]:
    ax.set_xlabel("Days since last gift")
for ax in axes[:, 0]:
    ax.set_ylabel("Lifetime giving (log scale)")
fig.suptitle("Each cluster (blue) against all other donors (gray)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 4. Clusters compared with rule-based segments

Each cell counts the donors in a cluster (row) who fall in a rule-based segment (column).

In [ ]:
compare_with_segments(df)

## 5. What the clusters show

With the committed seed-42 dataset:

| Cluster | Pattern | Possible fundraising action |
|---|---|---|
| 1 | Recent, frequent, highest giving (about 38% of income from 14 donors) | Stewardship and thank-you contact; upgrade asks |
| 2 | Steady givers with several gifts, last gift a few months ago | Keep engaged; regular-giving conversion |
| 3 | One or two gifts, last gift months ago or longer | Reactivation campaign before they lapse |
| 4 | Recent givers with only a few small gifts | Welcome journey to secure the next gift |

**Compared with the rules:**

- Cluster 3 combines most *Inactive* donors with most *New* donors. The rules call a one-gift donor "New" even if that gift was almost a year ago; the clustering groups those donors with the lapsing ones, which may be the more useful view for a retention campaign. Cluster 4 picks out the genuinely recent newcomers.
- Cluster 1 holds five of the eight *High Value* donors plus nine *Regular* donors who give often and recently but have not yet reached the $1,000 threshold. Those nine are natural candidates for an upgrade ask.

**Limitations:**

- The data is synthetic and small (100 donors), so the clusters show the method, not real donor behavior.
- K-Means assumes roughly round, similar-sized groups, and the choice of k is a judgment call here.
- Clusters describe past behavior; they do not predict who will give next. Any use with real donor data would need privacy review and human oversight.